# 01 — Exploração e auditoria do PlantVillage

Notebook pronto para executar no Google Colab.

Nesta etapa fazemos somente a **compreensão e auditoria do dataset**. Ainda não há split 70/15/15, data augmentation ou treinamento.


In [ ]:
from google.colab import drive

drive.mount("/content/drive")


In [ ]:
from pathlib import Path

BASE_DIR = Path("/content/drive/MyDrive/TCC")
DATA_DIR = BASE_DIR / "data"
RESULTS_DIR = BASE_DIR / "results"

ZIP_PATH = DATA_DIR / "data.zip"
LEAF_MAP_PATH = DATA_DIR / "leaf_grouping" / "leaf-map.json"
OUTPUT_CSV = RESULTS_DIR / "plantvillage_metadata_raw_color.csv"

DATA_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("ZIP:", ZIP_PATH)
print("Leaf map:", LEAF_MAP_PATH)


In [ ]:
!pip install -q huggingface_hub pandas


## Verificação dos arquivos

Se os arquivos já estiverem no Google Drive, o notebook não baixa novamente.


In [ ]:
from huggingface_hub import hf_hub_download

if not ZIP_PATH.exists():
    print("data.zip não encontrado. Baixando...")
    hf_hub_download(
        repo_id="mohanty/PlantVillage",
        filename="data.zip",
        repo_type="dataset",
        local_dir=str(DATA_DIR),
    )
else:
    print("data.zip já existe.")

if not LEAF_MAP_PATH.exists():
    print("leaf-map.json não encontrado. Baixando...")
    hf_hub_download(
        repo_id="mohanty/PlantVillage",
        filename="leaf_grouping/leaf-map.json",
        repo_type="dataset",
        local_dir=str(DATA_DIR),
    )
else:
    print("leaf-map.json já existe.")

print()
print("data.zip existe:", ZIP_PATH.exists())
print("leaf-map.json existe:", LEAF_MAP_PATH.exists())

if ZIP_PATH.exists():
    print(f"Tamanho do data.zip: {ZIP_PATH.stat().st_size / (1024 ** 3):.2f} GB")


## Preparar o módulo de auditoria

Esta célula usa a implementação produzida pelo Codex em `src/plantvillage_audit.py`, mas a grava temporariamente no ambiente do Colab para o notebook funcionar sozinho.


In [ ]:
MODULE_PATH = Path("/content/plantvillage_audit.py")
MODULE_PATH.write_text('"""Utilities for auditing PlantVillage metadata without extracting images."""\n\nfrom __future__ import annotations\n\nimport json\nimport zipfile\nfrom collections.abc import Iterable, Mapping, Sequence\nfrom pathlib import Path, PurePosixPath\nfrom typing import Any\n\nimport pandas as pd\n\n\nDEFAULT_IMAGE_EXTENSIONS = frozenset(\n    {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}\n)\nMISSING_LEAF_ID = "LEAF_ID_NOT_FOUND"\n\nMETADATA_COLUMNS = [\n    "zip_path",\n    "filename",\n    "classe",\n    "cultura",\n    "doenca",\n    "leaf_id",\n    "leaf_id_found",\n    "leaf_match_status",\n    "leaf_lookup_key",\n    "leaf_suggestions_count",\n]\n\n\ndef normalize_image_identifier(filename: str) -> str:\n    """Normalize an image filename using the PlantVillage loader logic."""\n    image_identifier = PurePosixPath(str(filename).replace("\\\\", "/")).name\n    image_identifier = image_identifier.replace("_final_masked", "")\n\n    if "___" in image_identifier:\n        image_identifier = image_identifier.split("___")[-1]\n\n    image_identifier = image_identifier.split("copy")[0]\n\n    suffix = PurePosixPath(image_identifier).suffix\n    if suffix:\n        image_identifier = image_identifier[: -len(suffix)]\n\n    return image_identifier.strip().lower()\n\n\ndef load_leaf_map(leaf_map_path: str | Path) -> dict[str, list[str]]:\n    """Load leaf-map.json and normalize keys for lookup."""\n    path = Path(leaf_map_path)\n    with path.open("r", encoding="utf-8") as file:\n        raw_leaf_map = json.load(file)\n\n    if not isinstance(raw_leaf_map, Mapping):\n        raise TypeError("leaf-map.json must contain a JSON object.")\n\n    leaf_map: dict[str, list[str]] = {}\n    for key, value in raw_leaf_map.items():\n        lookup_key = str(key).strip().lower()\n        leaf_map.setdefault(lookup_key, []).extend(_coerce_suggestions(value))\n\n    return leaf_map\n\n\ndef resolve_leaf_id(\n    filename: str,\n    classe: str,\n    leaf_map: Mapping[str, Sequence[str]],\n    missing_leaf_id: str = MISSING_LEAF_ID,\n) -> dict[str, Any]:\n    """Find the PlantVillage leaf_id for one image.\n\n    Missing or ambiguous matches are explicit; they are not converted into\n    fallback identifiers.\n    """\n    lookup_key = normalize_image_identifier(filename)\n    suggestions = leaf_map.get(lookup_key)\n\n    if suggestions is None:\n        return {\n            "leaf_id": missing_leaf_id,\n            "leaf_id_found": False,\n            "leaf_match_status": "not_found",\n            "leaf_lookup_key": lookup_key,\n            "leaf_suggestions_count": 0,\n        }\n\n    suggestions = _coerce_suggestions(suggestions)\n    if len(suggestions) == 1:\n        return {\n            "leaf_id": suggestions[0],\n            "leaf_id_found": True,\n            "leaf_match_status": "matched_unique",\n            "leaf_lookup_key": lookup_key,\n            "leaf_suggestions_count": 1,\n        }\n\n    if len(suggestions) > 1:\n        for suggestion in suggestions:\n            if classe in suggestion:\n                return {\n                    "leaf_id": suggestion,\n                    "leaf_id_found": True,\n                    "leaf_match_status": "matched_by_class",\n                    "leaf_lookup_key": lookup_key,\n                    "leaf_suggestions_count": len(suggestions),\n                }\n\n        return {\n            "leaf_id": missing_leaf_id,\n            "leaf_id_found": False,\n            "leaf_match_status": "ambiguous_no_class_match",\n            "leaf_lookup_key": lookup_key,\n            "leaf_suggestions_count": len(suggestions),\n        }\n\n    return {\n        "leaf_id": missing_leaf_id,\n        "leaf_id_found": False,\n        "leaf_match_status": "empty_suggestions",\n        "leaf_lookup_key": lookup_key,\n        "leaf_suggestions_count": 0,\n    }\n\n\ndef build_metadata_dataframe(\n    zip_path: str | Path,\n    leaf_map_path: str | Path,\n    image_extensions: Iterable[str] = DEFAULT_IMAGE_EXTENSIONS,\n) -> pd.DataFrame:\n    """Create one metadata row per image in raw/color/ inside data.zip."""\n    leaf_map = load_leaf_map(leaf_map_path)\n    extensions = {extension.lower() for extension in image_extensions}\n    records: list[dict[str, Any]] = []\n\n    with zipfile.ZipFile(zip_path, "r") as zip_file:\n        for member in zip_file.infolist():\n            if member.is_dir():\n                continue\n\n            path_info = _extract_raw_color_image_path(member.filename, extensions)\n            if path_info is None:\n                continue\n\n            classe = path_info["classe"]\n            cultura, doenca = split_class_name(classe)\n            leaf_info = resolve_leaf_id(path_info["filename"], classe, leaf_map)\n\n            records.append(\n                {\n                    "zip_path": path_info["zip_path"],\n                    "filename": path_info["filename"],\n                    "classe": classe,\n                    "cultura": cultura,\n                    "doenca": doenca,\n                    **leaf_info,\n                }\n            )\n\n    return pd.DataFrame.from_records(records, columns=METADATA_COLUMNS)\n\n\ndef audit_metadata(metadata: pd.DataFrame) -> dict[str, pd.DataFrame]:\n    """Summarize PlantVillage metadata quality and class distribution."""\n    _validate_columns(metadata)\n\n    total_images = int(len(metadata))\n    leaf_found = _as_bool_series(metadata["leaf_id_found"])\n    missing_leaf_count = int((~leaf_found).sum())\n    missing_leaf_percent = (\n        round((missing_leaf_count / total_images) * 100, 4) if total_images else 0.0\n    )\n\n    leaf_counts = metadata.loc[leaf_found].groupby("leaf_id").size()\n    leaves_with_multiple_images = leaf_counts[leaf_counts > 1]\n\n    diseases = metadata.loc[\n        ~metadata["doenca"].astype("string").str.lower().eq("healthy").fillna(False),\n        ["cultura", "doenca"],\n    ].drop_duplicates()\n\n    summary = pd.DataFrame(\n        [\n            {\n                "total_imagens": total_images,\n                "numero_classes": int(metadata["classe"].nunique(dropna=True)),\n                "numero_culturas": int(metadata["cultura"].nunique(dropna=True)),\n                "numero_doencas": int(len(diseases)),\n                "numero_leaf_id_unicos": int(metadata.loc[leaf_found, "leaf_id"].nunique()),\n                "imagens_sem_leaf_id": missing_leaf_count,\n                "percentual_sem_leaf_id": missing_leaf_percent,\n                "imagens_em_folhas_com_multiplas_imagens": int(\n                    leaves_with_multiple_images.sum()\n                ),\n                "folhas_com_multiplas_imagens": int(len(leaves_with_multiple_images)),\n            }\n        ]\n    )\n\n    class_counts = (\n        metadata["classe"]\n        .value_counts(dropna=False)\n        .sort_index()\n        .rename_axis("classe")\n        .reset_index(name="quantidade")\n    )\n\n    leaf_status_counts = (\n        metadata["leaf_match_status"]\n        .value_counts(dropna=False)\n        .sort_index()\n        .rename_axis("leaf_match_status")\n        .reset_index(name="quantidade")\n    )\n\n    return {\n        "resumo": summary,\n        "imagens_por_classe": class_counts,\n        "status_leaf_id": leaf_status_counts,\n    }\n\n\ndef save_metadata_csv(metadata: pd.DataFrame, output_path: str | Path) -> Path:\n    """Save metadata CSV and return its path."""\n    path = Path(output_path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    metadata.to_csv(path, index=False)\n    return path\n\n\ndef split_class_name(classe: str) -> tuple[str, str]:\n    """Split a PlantVillage class folder into crop and disease/condition."""\n    parts = str(classe).split("___", 1)\n    cultura = parts[0]\n    doenca = parts[1] if len(parts) > 1 else "unknown"\n    return cultura, doenca\n\n\ndef _coerce_suggestions(value: Any) -> list[str]:\n    if value is None:\n        return []\n\n    if isinstance(value, str):\n        return [value]\n\n    if isinstance(value, Sequence):\n        return [str(item) for item in value]\n\n    return [str(value)]\n\n\ndef _extract_raw_color_image_path(\n    zip_member_path: str,\n    image_extensions: set[str],\n) -> dict[str, str] | None:\n    normalized_path = str(zip_member_path).replace("\\\\", "/").lstrip("/")\n    parts = [part for part in normalized_path.split("/") if part]\n    lower_parts = [part.lower() for part in parts]\n\n    for index in range(len(parts) - 3):\n        if lower_parts[index] == "raw" and lower_parts[index + 1] == "color":\n            filename = parts[-1]\n            if PurePosixPath(filename).suffix.lower() not in image_extensions:\n                return None\n\n            return {\n                "zip_path": normalized_path,\n                "classe": parts[index + 2],\n                "filename": filename,\n            }\n\n    return None\n\n\ndef _validate_columns(metadata: pd.DataFrame) -> None:\n    missing_columns = [column for column in METADATA_COLUMNS if column not in metadata]\n    if missing_columns:\n        joined_columns = ", ".join(missing_columns)\n        raise ValueError(f"Missing required metadata columns: {joined_columns}")\n\n\ndef _as_bool_series(series: pd.Series) -> pd.Series:\n    if pd.api.types.is_bool_dtype(series):\n        return series.fillna(False).astype(bool)\n\n    normalized = series.astype("string").str.strip().str.lower()\n    return normalized.isin({"1", "true", "yes", "sim"}).fillna(False)\n\n\n__all__ = [\n    "DEFAULT_IMAGE_EXTENSIONS",\n    "METADATA_COLUMNS",\n    "MISSING_LEAF_ID",\n    "audit_metadata",\n    "build_metadata_dataframe",\n    "load_leaf_map",\n    "normalize_image_identifier",\n    "resolve_leaf_id",\n    "save_metadata_csv",\n    "split_class_name",\n]\n', encoding="utf-8")

print("Módulo preparado:", MODULE_PATH)


In [ ]:
import sys

if "/content" not in sys.path:
    sys.path.insert(0, "/content")

from plantvillage_audit import (
    audit_metadata,
    build_metadata_dataframe,
    save_metadata_csv,
)

print("Módulo importado com sucesso.")


## Construção dos metadados

O código percorre somente as imagens de `raw/color/` dentro do ZIP. Pode levar alguns minutos.


In [ ]:
metadata = build_metadata_dataframe(
    zip_path=ZIP_PATH,
    leaf_map_path=LEAF_MAP_PATH,
)

print("Formato do DataFrame:", metadata.shape)
metadata.head()


## Resumo da auditoria


In [ ]:
auditoria = audit_metadata(metadata)

auditoria["resumo"]


## Status da associação de `leaf_id`


In [ ]:
auditoria["status_leaf_id"]


## Quantidade de imagens por classe


In [ ]:
auditoria["imagens_por_classe"]


## Imagens sem `leaf_id`

Se esta tabela estiver vazia, todas as imagens foram associadas a uma folha.


In [ ]:
falhas_leaf_id = metadata.loc[
    ~metadata["leaf_id_found"],
    [
        "filename",
        "classe",
        "leaf_lookup_key",
        "leaf_match_status",
        "leaf_suggestions_count",
    ],
]

print("Quantidade de imagens sem leaf_id:", len(falhas_leaf_id))
falhas_leaf_id.head(10)


## Salvar CSV de metadados no Google Drive


In [ ]:
csv_path = save_metadata_csv(metadata, OUTPUT_CSV)

print("CSV salvo em:", csv_path)
print("Arquivo existe:", csv_path.exists())


## O que enviar para conferência

Depois de executar tudo, envie as saídas de:

- **Resumo da auditoria**
- **Status da associação de `leaf_id`**
- **Quantidade de imagens sem `leaf_id`**

A distribuição por classe também é útil, mas não é necessário copiar o CSV inteiro.
